## in this This approach I will use Cluster Based Embedding with attention mechanism to try to enhanced model
### To be Explained in Details inshallah in discussion
### regards.
#### Ahmed Fayad 

### Importing Libraries

In [53]:
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 500)

In [54]:
import matplotlib.pyplot as plt
import seaborn as sns

In [55]:
df = pd.read_csv(r"/kaggle/input/datasets/ahmedfayad/personallity-data/Lab 3 - Personality Profile Type.csv")
df.head(10)

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||http://41.media.tumblr.com/tumblr_lfouy03PMA1qa1rooo1_500.jpg|||enfp and intj moments https://www.youtube.com/watch?v=iz7lE1g4XM4 sportscenter not top ten plays https://www.youtube.com/watch?v=uCdfze1etec pranks|||What has been the most life-changing experience in your life?|||http://www.youtube.com/watch?v=vXZeYwwRDw8 http://www.youtube.com/watch?v=u8ejam5DP3E On repeat for most of today.|||May the PerC Experience immerse you.|||The last ...
1,ENTP,'I'm finding the lack of me in these posts very alarming.|||Sex can be boring if it's in the same position often. For example me and my girlfriend are currently in an environment where we have to creatively use cowgirl and missionary. There isn't enough...|||Giving new meaning to 'Game' theory.|||Hello *ENTP Grin* That's all it takes. Than we converse and they do most of the flirting while I acknowledge their presence and return their words with smooth wordplay and more cheeky grins.|||This...
2,INTP,"'Good one _____ https://www.youtube.com/watch?v=fHiGbolFFGw|||Of course, to which I say I know; that's my blessing and my curse.|||Does being absolutely positive that you and your best friend could be an amazing couple count? If so, than yes. Or it's more I could be madly in love in case I reconciled my feelings (which at...|||No, I didn't; thank you for a link!|||So-called Ti-Si loop (and it can stem from any current topic/obsession) can be deadly. It's like when you're stuck in your o..."
3,INTJ,"'Dear INTP, I enjoyed our conversation the other day. Esoteric gabbing about the nature of the universe and the idea that every rule and social code being arbitrary constructs created...|||Dear ENTJ sub, Long time no see. Sincerely, Alpha|||None of them. All other types hurt in deep existential ways that I want no part of.|||Probably a sliding scale that depends on individual preferences, like everything in humanity.|||Draco Malfoy also. I'd say he's either 358 or 368.|||I'm either 358..."
4,ENTJ,'You're fired.|||That's another silly misconception. That approaching is logically is going to be the key to unlocking whatever it is you think you are entitled to. Nobody wants to be approached with BS...|||But guys... he REALLY wants to go on a super-duper-long-ass vacation. C'mon guys. His boss just doesn't listen or get it. He even approached him logically and everything.|||Never mind. Just go on permanent vacation.|||Two months? I wouldn't be crazy about the idea. If you are really hi...
5,INTJ,"'18/37 @.@|||Science is not perfect. No scientist claims that it is, or that scientific information will not be revised as we discover new things. Rational thinking has been very useful to our society....|||INFP- Edgar Allen Poe was an INFP and he's in your siggy.|||People see the obvious Fi and are quick to put her as INFP. I agree that she has no Ne. I see her as an ISFP. Compare her to Haku (definite INFP). She is flat through most of Naruto.. but I don't...|||Lets get this party star..."
6,INFJ,"'No, I can't draw on my own nails (haha). Those were done by professionals on my nails. And yes, those are all gel. You mean those you posted were done by yourself on your own nails? Awesome!|||Probably the Electronic Screen Syndrome. With the advent of technology and social media, we all suffer from overstimulation on a daily basis. I'm guilty as well. In the past, I can be happy just...|||I love nail arts too! These are some of mine: 718282 718290 718298 718306 718314|||This is the first..."
7,INTJ,"'I tend to build up a collection of things on my desktop that i use frequently and then move them into a folder called 'Everything' from there it get sorted into type and sub type|||i ike to collect odd objects, even at work...a lot of people would call it junk but i like to collect it. Old unused software? ill take that off your hands :) i have a bunch of old adobe...|||i think its quite normal, i tend to only see my friends in real life every c

### Split Data

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
X = df.drop("type",axis = 1)
y = df["type"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,stratify=y, random_state=42)

In [6]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.0 MB/s eta 0:00:00


### Import nessacery modules to make predefiend cleaning steps in pervious notebook

In [7]:
import contractions
import re
import string

In [8]:
import spacy
import requests
from bs4 import BeautifulSoup
nlp = spacy.load("en_core_web_sm")
pd.set_option("display.max_rows", 200)

In [9]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

### Create Cleaning Function and apply this function at our Data

In [10]:
def clean_text_keep_separator(text: str):
    # 1. remove links
    text = re.sub(r'http\S+', '', text)
    # 2. remove special patterns except |||
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'\*\*\*', '', text)
    text = re.sub(r'//', '', text)
    # 3. remove numbers
    text = re.sub(r'\d+', '', text)
    # 4. fix contractions
    text = contractions.fix(text)
    # 5. lowercase
    text = text.lower()
    # 6. remove punctuation except |||
    # Replace ||| temporarily with placeholder
    text = text.replace("|||", "SPECIALSEP")
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Restore |||
    text = text.replace("SPECIALSEP", "|||")
    # 7. tokenize, lemmatize, and rejoin
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    text = ' '.join(lemmatized_words)
    return text

In [13]:
from tqdm import tqdm
tqdm.pandas()

X_train = X_train["posts"].progress_apply(clean_text_keep_separator)

100%|██████████| 6940/6940 [00:36<00:00, 190.86it/s]


In [14]:
from tqdm import tqdm
tqdm.pandas()

X_test = X_test["posts"].astype(str).progress_apply(clean_text_keep_separator)

100%|██████████| 1735/1735 [00:08<00:00, 210.15it/s]


## Modeling step

### Importing neccesary library to make model using pytorch

In [15]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import os, requests, zipfile
from sklearn.metrics import f1_score

### define Some configs

In [42]:
# ─────────────────────────────────────────────────────────────────────
# 0. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
CONFIG = {
    "glove_dim":    100,        # GloVe vector size
    "lstm_hidden":  256,        # increased from 128 → more capacity
    "num_layers":   2,          # 2-layer LSTM → deeper sequential modeling
    "fc_hidden":    256,        # vanilla NN hidden size
    "dropout":      0.3,
    "num_posts":    50,
    "max_words":    200,        # increased from 100 → less signal thrown away
    "batch_size":   32,
    "epochs":       20,
    "lr":           1e-3,
    "aggregation":  "attention",     # "mean" | "attention"
    "glove_path":   "glove.6B.100d.txt",
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
    "proto_lambda": 0.5,        # weight of alignment loss relative to BCE loss
    "proto_freeze": 3,          # epochs to freeze prototypes (let BiLSTM warm up first)
}
 
MBTI_DIMS = [
    {"name": "E/I", "pos": "E", "neg": "I"},
    {"name": "N/S", "pos": "N", "neg": "S"},
    {"name": "T/F", "pos": "T", "neg": "F"},
    {"name": "J/P", "pos": "J", "neg": "P"},
]

### Make Some helper Functions 

In [43]:
# ─────────────────────────────────────────────────────────────────────
# 1. LABEL ENCODING / DECODING
# ─────────────────────────────────────────────────────────────────────
def encode_mbti(mbti_str):
    """'INFJ' → [1, 0, 0, 0]"""
    mbti_str = mbti_str.strip().upper()
    return [0.0 if dim["pos"] in mbti_str else 1.0 for dim in MBTI_DIMS]
 
def decode_mbti(preds):
    """[1, 0, 0, 0] → 'INFJ'"""
    return "".join(dim["pos"] if int(p) == 0 else dim["neg"]
                   for dim, p in zip(MBTI_DIMS, preds))
 

In [44]:
# ─────────────────────────────────────────────────────────────────────
# 2. VOCABULARY
# ─────────────────────────────────────────────────────────────────────
def build_vocab(X, min_freq=2):
    counter = Counter()
    for instance in X:
        for post in str(instance).split("|||"):
            counter.update(post.strip().split())
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    print(f"[Vocab] Size: {len(vocab):,}")
    return vocab
 
 
# 

In [45]:
# ─────────────────────────────────────────────────────────────────────
# 3. GLOVE
# ─────────────────────────────────────────────────────────────────────
def download_glove(path, dim=100):
    if os.path.exists(path):
        print(f"[GloVe] Found: {path}"); return
    print("[GloVe] Downloading...")
    zip_path = "glove.6B.zip"
    with requests.get("https://nlp.stanford.edu/data/glove.6B.zip", stream=True) as r:
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(8192): f.write(chunk)
    with zipfile.ZipFile(zip_path) as z:
        z.extract(f"glove.6B.{dim}d.txt")
    os.remove(zip_path)
    print("[GloVe] Done.")
 
def load_glove(path, vocab, dim=100):
    download_glove(path, dim)
    glove = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if parts[0] in vocab:
                glove[parts[0]] = np.array(parts[1:], dtype=np.float32)
    matrix = np.random.uniform(-0.1, 0.1, (len(vocab), dim)).astype(np.float32)
    matrix[0] = 0.0     # <PAD> → zero
    found = 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]; found += 1
    print(f"[GloVe] Coverage: {found:,}/{len(vocab):,} ({found/len(vocab)*100:.1f}%)")
    return matrix

### Make Class to Preprocess dataset

In [46]:
# ─────────────────────────────────────────────────────────────────────
# 4. DATASET
#    Tokenises each post and pads to max_words.
#    Returns token indices of shape (num_posts, max_words).
# ─────────────────────────────────────────────────────────────────────
class MBTIDataset(Dataset):
    def __init__(self, X, y, vocab, config):
        self.data   = []
        max_words   = config["max_words"]
        num_posts   = config["num_posts"]
        pad_idx     = vocab["<PAD>"]
        unk_idx     = vocab["<UNK>"]
 
        for instance_str, label_str in zip(X, y):
            posts = str(instance_str).split("|||")[:num_posts]
            while len(posts) < num_posts:
                posts.append("")                            # pad to 50 posts
 
            encoded_posts = []
            for post in posts:
                tokens  = post.strip().split()             # already cleaned
                indices = [vocab.get(t, unk_idx) for t in tokens]
                # pad / truncate to max_words
                indices = indices[:max_words]
                indices += [pad_idx] * (max_words - len(indices))
                encoded_posts.append(indices)
 
            x_tensor = torch.tensor(encoded_posts, dtype=torch.long)   # (num_posts, max_words)
            y_tensor = torch.tensor(encode_mbti(str(label_str)), dtype=torch.float32)
            self.data.append((x_tensor, y_tensor))
 
    def __len__(self):  return len(self.data)
    def __getitem__(self, i): return self.data[i]
 
 
# 

### Create model Class 

In [56]:
# ─────────────────────────────────────────────────────────────────────
# 5. MODEL
# ─────────────────────────────────────────────────────────────────────
class AttentionAggregator(nn.Module):
    """
    Learns a scalar weight for each of the 50 post vectors,
    then returns their weighted sum as the person vector.
    Alternative to simple mean pooling.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.attn = nn.Linear(input_dim, 1)
 
    def forward(self, post_vecs):
        """
        post_vecs : (batch, num_posts, input_dim)
        returns   : (batch, input_dim)
        """
        scores  = self.attn(post_vecs).squeeze(-1)      # (batch, num_posts)
        weights = torch.softmax(scores, dim=-1)          # (batch, num_posts)
        out     = (weights.unsqueeze(-1) * post_vecs).sum(dim=1)  # (batch, input_dim)
        return out
 
 
class ClusterAttention(nn.Module):
    """
    16 learnable prototype vectors, one per MBTI type.
    The person vector (query) attends over all 16 prototypes (keys/values).
    Output: a cluster_context vector = weighted sum of prototypes.
 
    This forces the model to learn what each MBTI type looks like as a
    dense vector, and lets each person softly align to the most similar types.
 
    Shapes:
        person_vec    : (batch, person_dim)
        prototypes    : (16, person_dim)          learnable
        attn_scores   : (batch, 16)               similarity to each type
        cluster_ctx   : (batch, person_dim)       weighted prototype mix
    """
    NUM_TYPES = 16  # MBTI has 16 possible types
 
    def __init__(self, person_dim):
        super().__init__()
        # 16 learnable prototype vectors, one per MBTI type
        self.prototypes = nn.Parameter(
            torch.randn(self.NUM_TYPES, person_dim) * 0.01
        )
        # Project person_vec into query space
        self.query_proj = nn.Linear(person_dim, person_dim, bias=False)
        # Scale factor for dot-product attention (sqrt of dim)
        self.scale = person_dim ** 0.5
 
    def forward(self, person_vec):
        """
        person_vec : (batch, person_dim)
        returns    : cluster_ctx (batch, person_dim),
                     attn_weights (batch, 16)  -- kept for inspection
        """
        # Project person_vec to query
        query = self.query_proj(person_vec)             # (batch, person_dim)
 
        # Dot-product attention: query vs all 16 prototypes
        # prototypes : (16, person_dim) -> transpose to (person_dim, 16)
        scores  = torch.matmul(query, self.prototypes.T) / self.scale
        # scores : (batch, 16)
 
        weights = torch.softmax(scores, dim=-1)         # (batch, 16)
 
        # Weighted sum of prototypes
        cluster_ctx = torch.matmul(weights, self.prototypes)
        # cluster_ctx : (batch, person_dim)
 
        return cluster_ctx, weights

### Create Custom API to run in Our Dataset

In [48]:
class MBTIClassifier(nn.Module):
    """
    ┌─────────────────────────────────────────────────────┐
    │  Input: (batch, 50 posts, max_words)                │
    │                                                     │
    │  LEVEL 1 — Word LSTM (per post)                     │
    │    GloVe embed  →  (batch*50, max_words, glove_dim) │
    │    LSTM         →  last hidden state                │
    │    Result       →  (batch, 50, lstm_hidden)         │
    │                                                     │
    │  LEVEL 2 — Aggregation (50 → 1 vector per person)  │
    │    mean or attention pooling                        │
    │    Result       →  (batch, lstm_hidden)             │
    │                                                     │
    │  LEVEL 3 — Vanilla NN                               │
    │    FC → ReLU → Dropout → FC → 4 logits             │
    │    Sigmoid applied at inference (BCEWithLogits      │
    │    handles it during training)                      │
    └─────────────────────────────────────────────────────┘
    """
    def __init__(self, vocab_size, embedding_matrix, config):
        super().__init__()
 
        glove_dim   = config["glove_dim"]
        lstm_hidden = config["lstm_hidden"]
        fc_hidden   = config["fc_hidden"]
        dropout     = config["dropout"]
        self.num_posts  = config["num_posts"]
        self.max_words  = config["max_words"]
        self.aggregation = config["aggregation"]
 
        # ── GloVe Embedding (trainable) ──────────────────────────────────
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=glove_dim,
            padding_idx=0,                  # PAD token → zero vector, no gradient
        )
        self.embedding.weight = nn.Parameter(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            requires_grad=True              # fine-tune during training
        )
 
        # ── Level 1: Word-level LSTM ─────────────────────────────────────
        # Reads word embeddings in a post sequentially.
        # Last hidden state = 1 vector summarising the post.
        self.word_lstm = nn.LSTM(
            input_size=glove_dim,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,             # forward + backward over words in each post
        )
 
        # ── Level 2: Aggregation (50 post vectors → 1 person vector) ─────
        if self.aggregation == "attention":
            self.aggregator = AttentionAggregator(lstm_hidden * 2)
        # "mean" needs no parameters
 
        # Level 2.5: Cluster Attention
        # person_vec (256d) attends over 16 MBTI prototypes -> cluster_context
        person_dim = lstm_hidden * 2
        self.cluster_attn = ClusterAttention(person_dim)
 
        # ── Level 3: Vanilla NN ──────────────────────────────────────────
        # Simple feed-forward network ending in 4 outputs.
        # BCEWithLogitsLoss applies sigmoid during training.
        self.classifier = nn.Sequential(
            nn.Linear(person_dim * 2, fc_hidden),  # 512 -> fc_hidden (cat of person_vec + cluster_ctx)
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_hidden, 4),        # 4 raw logits (one per MBTI dim)
        )
 
    def forward(self, x):
        """
        x : (batch, num_posts, max_words)
        """
        batch_size  = x.shape[0]
 
        # ── Level 1: embed words + run LSTM per post ──────────────────────
        # Flatten posts into one batch dimension for efficiency
        x_flat   = x.view(batch_size * self.num_posts, self.max_words)
        # x_flat : (batch*50, max_words)
 
        embedded = self.embedding(x_flat)
        # embedded : (batch*50, max_words, glove_dim)
 
        _, (h_n, _) = self.word_lstm(embedded)
        # h_n : (2, batch*50, lstm_hidden)  -- 2 because bidirectional
        # Concatenate forward and backward final hidden states
        post_vecs = torch.cat([h_n[0], h_n[1]], dim=-1)
        # post_vecs : (batch*50, lstm_hidden*2)
 
        # Reshape back to separate batch and posts
        post_vecs = post_vecs.view(batch_size, self.num_posts, -1)
        # post_vecs : (batch, 50, lstm_hidden)
        # ← these are the 50 vectors per person, each summarising one post
 
        # ── Level 2: aggregate 50 → 1 person vector ───────────────────────
        # Level 2: aggregate 50 post vectors -> 1 person vector
        if self.aggregation == "attention":
            person_vec = self.aggregator(post_vecs)     # (batch, 256)
        else:
            person_vec = post_vecs.mean(dim=1)          # (batch, 256)
        # person_vec: (batch, lstm_hidden*2 = 256)
 
        # Level 2.5: Cluster Attention
        # person_vec queries all 16 MBTI prototype vectors
        cluster_ctx, attn_weights = self.cluster_attn(person_vec)
        # cluster_ctx  : (batch, 256) - weighted mix of prototypes
        # attn_weights : (batch, 16)  - how much each type contributed
 
        # Concatenate person vector with cluster context
        enriched = torch.cat([person_vec, cluster_ctx], dim=-1)
        # enriched : (batch, 512)
 
        # Level 3: Vanilla NN
        logits = self.classifier(enriched)              # (batch, 4)
        return logits
 
# 

### Training And Evoultion Function

In [49]:
# ─────────────────────────────────────────────────────────────────────
# 6. TRAINING & EVALUATION
# ─────────────────────────────────────────────────────────────────────
def compute_pos_weights(y_train, device):
    """Compute per-dimension class weights to handle imbalanced MBTI data."""
    labels    = torch.tensor([encode_mbti(str(l)) for l in y_train])
    pos       = labels.sum(dim=0)
    neg       = len(labels) - pos
    return (neg / pos.clamp(min=1)).to(device)
 
 
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)
 
 
@torch.no_grad()
def evaluate(model, loader, criterion, device, thresholds=None):
    """
    thresholds : list of 4 floats, one per dimension.
                 Defaults to 0.5 for all if not provided.
    """
    if thresholds is None:
        thresholds = [0.5] * 4
 
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []
 
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        logits      = model(x_batch)
        total_loss += criterion(logits, y_batch).item()
        all_probs.append(torch.sigmoid(logits).cpu())
        all_labels.append(y_batch.int().cpu())
 
    probs  = torch.cat(all_probs)                       # (N, 4)
    labels = torch.cat(all_labels)                      # (N, 4)
 
    # Apply per-dimension thresholds
    thresh_t = torch.tensor(thresholds)
    preds    = (probs > thresh_t).int()                 # (N, 4)
 
    # Per-dimension accuracy
    dim_acc  = (preds == labels).float().mean(dim=0).tolist()
 
    # Full type accuracy (all 4 correct)
    full_acc = (preds == labels).all(dim=1).float().mean().item()
 
    # ≥3/4 correct accuracy (per requirement)
    correct_dims = (preds == labels).sum(dim=1)         # (N,) values 0-4
    partial_acc  = (correct_dims >= 3).float().mean().item()
 
    return total_loss / len(loader), dim_acc, full_acc, partial_acc, probs.numpy(), labels.numpy()
 
 
def find_best_thresholds(probs, labels):
    """Find the F1-optimal threshold per dimension on a held-out set."""
    best_thresholds = []
    dim_names = [d["name"] for d in MBTI_DIMS]
    print("\n[Threshold tuning]")
    for i, name in enumerate(dim_names):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.30, 0.71, 0.02):
            f1 = f1_score(labels[:, i], (probs[:, i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds.append(float(best_t))
        print(f"  {name}: threshold = {best_t:.2f}  F1 = {best_f1:.3f}")
    return best_thresholds

### Make unified Run Function 

In [50]:
# ─────────────────────────────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────────────────────────────
def run(X_train, y_train, X_test, y_test):
    """
    X_train / X_test : list or pd.Series — strings with posts separated by '|||'
    y_train / y_test : list or pd.Series — MBTI strings e.g. 'INFJ'
    """
    device = CONFIG["device"]
    print(f"[Device] {device.upper()}")
    print(f"[Data]   Train: {len(X_train):,} | Test: {len(X_test):,}")
 
    # ── Vocab + GloVe ────────────────────────────────────────────────
    vocab      = build_vocab(list(X_train) + list(X_test))
    emb_matrix = load_glove(CONFIG["glove_path"], vocab, CONFIG["glove_dim"])
 
    # ── Datasets ─────────────────────────────────────────────────────
    train_ds = MBTIDataset(X_train, y_train, vocab, CONFIG)
    test_ds  = MBTIDataset(X_test,  y_test,  vocab, CONFIG)
 
    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False)
 
    # ── Model ────────────────────────────────────────────────────────
    model = MBTIClassifier(len(vocab), emb_matrix, CONFIG).to(device)
    print(f"[Model]  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"[Model]  Aggregation: {CONFIG['aggregation']}")
 
    # ── Loss (with class weights for imbalanced MBTI data) ───────────
    pos_weight = compute_pos_weights(y_train, device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
 
    optimizer  = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
 
    dim_names      = [d["name"] for d in MBTI_DIMS]
    best_test_loss = float("inf")
 
    # ── Training loop ────────────────────────────────────────────────
    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, dim_acc, full_acc, partial_acc, _, _ = evaluate(
            model, test_loader, criterion, device
        )
        scheduler.step(test_loss)
 
        acc_str = " | ".join(f"{n}: {a*100:.1f}%" for n, a in zip(dim_names, dim_acc))
        print(
            f"Epoch {epoch:02d}/{CONFIG['epochs']} "
            f"| Train: {train_loss:.4f} | Test: {test_loss:.4f} "
            f"| ≥3/4 Acc: {partial_acc*100:.2f}% "
            f"| Full Acc: {full_acc*100:.2f}% "
            f"| [{acc_str}]"
        )
 
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            torch.save(model.state_dict(), "best_mbti_model.pt")
            print("  ✓ Saved.")
 
    # ── Load best and tune thresholds ────────────────────────────────
    model.load_state_dict(torch.load("best_mbti_model.pt"))
    _, _, _, _, probs, labels = evaluate(model, test_loader, criterion, device)
    best_thresholds = find_best_thresholds(probs, labels)
 
    # ── Final evaluation with tuned thresholds ───────────────────────
    test_loss, dim_acc, full_acc, partial_acc, _, _ = evaluate(
        model, test_loader, criterion, device, thresholds=best_thresholds
    )
 
    print("\n" + "=" * 60)
    print("FINAL TEST RESULTS (with tuned thresholds)")
    print("=" * 60)
    print(f"  Test Loss      : {test_loss:.4f}")
    print(f"  ≥3/4 Acc       : {partial_acc * 100:.2f}%   ← requirement metric")
    print(f"  Full Type Acc  : {full_acc * 100:.2f}%")
    for name, acc in zip(dim_names, dim_acc):
        print(f"  {name} Accuracy   : {acc * 100:.2f}%")
 
    return model, vocab, best_thresholds

### Function to make Inference

In [51]:
# ─────────────────────────────────────────────────────────────────────
# 8. INFERENCE
# ─────────────────────────────────────────────────────────────────────
def predict(model, vocab, instance_str, thresholds=None, device=CONFIG["device"]):
    """
    instance_str : string of posts separated by '|||', already cleaned
    thresholds   : list of 4 floats from find_best_thresholds() (optional)
    """
    if thresholds is None:
        thresholds = [0.5] * 4
 
    config   = CONFIG
    pad_idx  = vocab["<PAD>"]
    unk_idx  = vocab["<UNK>"]
    posts    = str(instance_str).split("|||")[:config["num_posts"]]
    while len(posts) < config["num_posts"]:
        posts.append("")
 
    encoded = []
    for post in posts:
        tokens  = post.strip().split()
        indices = [vocab.get(t, unk_idx) for t in tokens]
        indices = indices[:config["max_words"]]
        indices += [pad_idx] * (config["max_words"] - len(indices))
        encoded.append(indices)
 
    x = torch.tensor([encoded], dtype=torch.long).to(device)   # (1, 50, max_words)
 
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs  = torch.sigmoid(logits).squeeze().tolist()
        preds  = [int(p > t) for p, t in zip(probs, thresholds)]
 
    return decode_mbti(preds)

In [52]:
# ─────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    model, vocab, thresholds = run(X_train, y_train, X_test, y_test)
    # pred = predict(model, vocab, X_test.iloc[0], thresholds)
    pass

[Device] CUDA
[Data]   Train: 6,940 | Test: 1,735
[Vocab] Size: 54,064
[GloVe] Found: glove.6B.100d.txt
[GloVe] Coverage: 39,893/54,064 (73.8%)
[Model]  Parameters: 6,673,861
[Model]  Aggregation: attention
Epoch 01/20 | Train: 0.6700 | Test: 0.6590 | ≥3/4 Acc: 32.91% | Full Acc: 4.50% | [E/I: 23.1% | N/S: 72.8% | T/F: 64.8% | J/P: 51.5%]
  ✓ Saved.
Epoch 02/20 | Train: 0.6228 | Test: 0.5974 | ≥3/4 Acc: 45.19% | Full Acc: 9.16% | [E/I: 46.8% | N/S: 49.8% | T/F: 82.7% | J/P: 54.6%]
  ✓ Saved.
Epoch 03/20 | Train: 0.5298 | Test: 0.5367 | ≥3/4 Acc: 54.70% | Full Acc: 15.27% | [E/I: 54.8% | N/S: 63.2% | T/F: 82.8% | J/P: 53.1%]
  ✓ Saved.
Epoch 04/20 | Train: 0.4003 | Test: 0.5258 | ≥3/4 Acc: 60.17% | Full Acc: 18.33% | [E/I: 52.0% | N/S: 81.8% | T/F: 82.9% | J/P: 50.7%]
  ✓ Saved.
Epoch 05/20 | Train: 0.3039 | Test: 0.5769 | ≥3/4 Acc: 66.63% | Full Acc: 20.29% | [E/I: 66.7% | N/S: 86.2% | T/F: 83.4% | J/P: 41.7%]
Epoch 06/20 | Train: 0.2774 | Test: 0.6133 | ≥3/4 Acc: 69.51% | Full Acc: 27